In [0]:
%python
from pyspark.sql.functions import col, sum, count, avg, round, year, month, when, countDistinct, dense_rank, row_number, desc, asc
from pyspark.sql.window import Window


In [0]:
--  Customer Loyalty - Repeat customer rate by continent

WITH customer_orders AS (
  SELECT 
    c.continent,
    f.customer_key,
    COUNT(DISTINCT f.order_number) AS order_count
  FROM electronic_retailer.03_Gold.fact_sales f
  JOIN electronic_retailer.03_Gold.dim_customers c ON f.customer_key = c.customer_key
  GROUP BY c.continent, f.customer_key
)
SELECT 
  continent,
  COUNT(customer_key) AS unique_customers,
  SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) AS repeat_customers,
  ROUND(SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(customer_key), 2) AS repeat_percentage
FROM customer_orders
GROUP BY continent
ORDER BY repeat_percentage DESC

In [0]:
-- Customer count and spending by continent and gender

WITH customer_counts AS (
  SELECT 
    c.continent,
    c.gender,
    COUNT(DISTINCT f.customer_key) AS customer_count
  FROM electronic_retailer.03_Gold.fact_sales f
  JOIN electronic_retailer.03_Gold.dim_customers c ON f.customer_key = c.customer_key
  GROUP BY c.continent, c.gender
),
revenue_totals AS (
  SELECT 
    customer_continent AS continent,
    gender,
    ROUND(SUM(total_revenue), 2) AS total_spend_usd
  FROM electronic_retailer.03_Gold.data_cube_sales
  GROUP BY customer_continent, gender
)
SELECT 
  cc.continent,
  cc.gender,
  cc.customer_count,
  rt.total_spend_usd,
  ROUND(rt.total_spend_usd / cc.customer_count, 2) AS avg_spend_per_cust
FROM customer_counts cc
JOIN revenue_totals rt ON cc.continent = rt.continent AND cc.gender = rt.gender
ORDER BY cc.continent, cc.gender

In [0]:
--  Top 5 categories by revenue

WITH category_revenue AS (
  SELECT 
    category,
    ROUND(SUM(total_revenue), 2) AS revenue_usd
  FROM electronic_retailer.03_Gold.data_cube_sales
  GROUP BY category
),
total AS (
  SELECT SUM(revenue_usd) AS total_revenue FROM category_revenue
)
SELECT 
  category,
  revenue_usd,
  ROUND((revenue_usd / total_revenue) * 100, 2) AS pct_of_total_revenue
FROM category_revenue, total
ORDER BY revenue_usd DESC
LIMIT 5

In [0]:
-- KPI 7: Top 5 categories by units sold

WITH category_units AS (
  SELECT 
    category,
    SUM(total_quantity) AS units_sold
  FROM electronic_retailer.03_Gold.data_cube_sales
  GROUP BY category
),
total AS (
  SELECT SUM(units_sold) AS total_units FROM category_units
)
SELECT 
  category,
  units_sold,
  ROUND((units_sold / total_units) * 100, 2) AS percentage
FROM category_units, total
ORDER BY units_sold DESC
LIMIT 5

In [0]:
WITH channel_aov AS (
  SELECT 
    customer_continent AS continent,
    CASE WHEN store_country IS NULL THEN 'Online' ELSE 'In-Store' END AS channel,
    ROUND(SUM(total_revenue) / SUM(order_count), 2) AS aov,
    SUM(order_count) AS orders
  FROM electronic_retailer.03_Gold.data_cube_sales
  GROUP BY customer_continent, CASE WHEN store_country IS NULL THEN 'Online' ELSE 'In-Store' END
)
SELECT 
  continent,
  MAX(CASE WHEN channel = 'Online' THEN aov END) AS aov_online,
  MAX(CASE WHEN channel = 'Online' THEN orders END) AS online_orders,
  MAX(CASE WHEN channel = 'In-Store' THEN aov END) AS aov_store,
  MAX(CASE WHEN channel = 'In-Store' THEN orders END) AS store_orders
FROM channel_aov
GROUP BY continent
ORDER BY continent

In [0]:
--  Slowest 5 countries by delivery time
SELECT 
  customer_country AS country,
  ROUND(SUM(avg_delivery_days * order_count) / SUM(order_count), 2) AS avg_days,
  SUM(order_count) AS order_count
FROM electronic_retailer.03_Gold.data_cube_sales
WHERE avg_delivery_days IS NOT NULL
GROUP BY customer_country
ORDER BY avg_days DESC


In [0]:
-- overall avg delivery time across all orders
SELECT 
  ROUND(SUM(avg_delivery_days * order_count) / SUM(order_count), 2) AS average_days,
  SUM(order_count) AS total_orders
FROM electronic_retailer.03_Gold.data_cube_sales
WHERE avg_delivery_days IS NOT NULL

In [0]:
-- top 3 product categories driving revenue
WITH q2_revenue AS (
  SELECT 
    category,
    ROUND(SUM(total_revenue), 2) AS peak_month_revenue
  FROM electronic_retailer.03_Gold.data_cube_sales
  WHERE year = 2016 AND month IN (4, 5, 6)
  GROUP BY category
),
total AS (
  SELECT SUM(peak_month_revenue) AS q2_total FROM q2_revenue
)
SELECT 
  category,
  peak_month_revenue,
  ROUND((peak_month_revenue / q2_total) * 100, 2) AS pct_of_peak_total
FROM q2_revenue, total
ORDER BY peak_month_revenue DESC
LIMIT 3

In [0]:
-- Top 3 revenue months with percentage

WITH monthly_revenue AS (
  SELECT 
    month,
    month_name,
    ROUND(SUM(total_revenue), 2) AS revenue_usd
  FROM electronic_retailer.03_Gold.data_cube_sales
  WHERE year = 2016
  GROUP BY month, month_name
),
total AS (
  SELECT SUM(revenue_usd) AS total_revenue FROM monthly_revenue
)
SELECT 
  month,
  month_name,
  revenue_usd,
  ROUND((revenue_usd / total_revenue) * 100, 2) AS percentage
FROM monthly_revenue, total
ORDER BY revenue_usd DESC
LIMIT 3

In [0]:
-- Monthly revenue 
SELECT 
  year,
  month,
  month_name,
  ROUND(SUM(total_revenue), 2) AS revenue_usd
FROM electronic_retailer.03_Gold.data_cube_sales
WHERE year = 2016
GROUP BY year, month, month_name
ORDER BY month